# Time Series Track · Deep Learning for Time Series

**Track:** Traditional AI Domains — Time Series  
**Prerequisites:** TS/01-02, DL/01 (Neural Network Math), NB10b (PyTorch)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 120–150 minutes

---

### 🔍 Socratic Deep Check
*When should you use an LSTM vs a Temporal Convolutional Network vs a Transformer for time series — and why are foundation models (Chronos, TimesFM) changing everything?*

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Deep learning for time series only makes sense when you have (1) large datasets with many related series, (2) complex non-linear dynamics, or (3) multivariate inputs. For a single univariate series with 500 points, ARIMA or XGBoost wins. Seniors match architecture to data characteristics.

**Mental Model:** LSTMs read one word at a time (sequential, slow). TCNs read paragraphs using a sliding window (parallelizable, fast). Transformers read the entire book at once (attention, most flexible). Foundation models have already read every book in the library (pre-trained, zero-shot).

**Common Junior Pitfall:** Using a Transformer with 50M parameters on a 1000-point univariate series. The model memorizes noise and forecasts terribly. Deep learning needs thousands of related time series (e.g., all products in a store) to learn transferable patterns.

---

## 📑 Table of Contents

  * [🔍 Socratic Deep Check](#socratic-deep-check)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · LSTM for Time Series](#1-lstm-for-time-series)
* [2 · Temporal Convolutional Network (TCN)](#2-temporal-convolutional-network-tcn)
* [3 · Transformer for Time Series](#3-transformer-for-time-series)
* [4 · Foundation Models: Chronos & TimesFM](#4-foundation-models-chronos-timesfm)
* [Knowledge Check](#knowledge-check)
  * [Q1: LSTM vs TCN](#q1-lstm-vs-tcn)
  * [Q2: Why use a causal mask in the Transformer?](#q2-why-use-a-causal-mask-in-the-transformer)


In [ ]:
!pip install -q torch numpy pandas matplotlib scikit-learn

In [ ]:
# Cell 1 — Create windowed dataset for sequence models
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

np.random.seed(42)
n = 365 * 3
t = np.arange(n)
values = (np.linspace(100, 180, n) + 20*np.sin(2*np.pi*t/365) + 
          5*np.sin(2*np.pi*t/7) + np.random.normal(0, 5, n) +
          np.where(t > 600, 30, 0))

# Normalize
mean, std = values[:int(0.8*n)].mean(), values[:int(0.8*n)].std()
normalized = (values - mean) / std

def create_sequences(data, seq_len=30, horizon=1):
    """Create (input_window, target) pairs for sequence models."""
    X, y = [], []
    for i in range(len(data) - seq_len - horizon + 1):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len:i+seq_len+horizon])
    return np.array(X), np.array(y)

SEQ_LEN = 30
X, y = create_sequences(normalized, seq_len=SEQ_LEN, horizon=1)

# Temporal split
split = int(0.8 * len(X))
X_train, X_test = torch.FloatTensor(X[:split]).unsqueeze(-1), torch.FloatTensor(X[split:]).unsqueeze(-1)
y_train, y_test = torch.FloatTensor(y[:split]), torch.FloatTensor(y[split:])

print(f'X_train: {X_train.shape}  (samples, seq_len, features)')
print(f'y_train: {y_train.shape}  (samples, horizon)')
print(f'X_test:  {X_test.shape}')

---
## 1 · LSTM for Time Series

LSTMs process sequences step-by-step, maintaining a hidden state ("memory") that captures long-term patterns.

```
Input:  [x₁, x₂, x₃, ..., x₃₀]  (30-day window)
                    ↓
LSTM:   h₁ → h₂ → h₃ → ... → h₃₀  (hidden states carry memory)
                                ↓
Output: ŷ₃₁                     (predict day 31)
```

In [ ]:
# Cell 2 — LSTM Model

class LSTMForecaster(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, output_dim=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, 
                           batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # x: (batch, seq_len, features)
        lstm_out, (h_n, c_n) = self.lstm(x)  # h_n: final hidden state
        return self.fc(h_n[-1])                # Use last layer's hidden state

# Train
lstm_model = LSTMForecaster()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
criterion = nn.MSELoss()

dataset = torch.utils.data.TensorDataset(X_train, y_train)
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

losses = []
for epoch in range(30):
    epoch_loss = 0
    for xb, yb in loader:
        pred = lstm_model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(loader))
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d} | Loss: {losses[-1]:.6f}')

# Evaluate
lstm_model.eval()
with torch.no_grad():
    preds = lstm_model(X_test).numpy() * std + mean
    actuals = y_test.numpy() * std + mean
    mae = np.abs(preds - actuals).mean()
    print(f'\nLSTM Test MAE: {mae:.2f}')

---
## 2 · Temporal Convolutional Network (TCN)

TCNs use **dilated causal convolutions** — they're parallelizable (faster than LSTMs) and handle long-range dependencies via exponentially increasing dilation.

```
Dilation = 1:  [█ █ █]  → sees 3 timesteps
Dilation = 2:  [█ . █ . █]  → sees 5 timesteps  
Dilation = 4:  [█ . . . █ . . . █]  → sees 9 timesteps
Stacked: receptive field grows exponentially!
```

In [ ]:
# Cell 3 — TCN Model

class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=padding)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    
    def forward(self, x):
        out = self.dropout(self.relu(self.conv(x)))
        out = out[:, :, :x.size(2)]  # Causal: trim future padding
        return self.relu(out + self.residual(x))

class TCNForecaster(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=32, num_layers=4, output_dim=1):
        super().__init__()
        layers = [TCNBlock(input_dim, hidden_dim, dilation=1)]
        for i in range(1, num_layers):
            layers.append(TCNBlock(hidden_dim, hidden_dim, dilation=2**i))
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # x: (batch, seq_len, features) → conv expects (batch, features, seq_len)
        out = self.tcn(x.permute(0, 2, 1))
        return self.fc(out[:, :, -1])  # Last timestep

# Train TCN
tcn_model = TCNForecaster()
optimizer = torch.optim.Adam(tcn_model.parameters(), lr=0.001)

for epoch in range(30):
    epoch_loss = 0
    for xb, yb in loader:
        pred = tcn_model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d} | Loss: {epoch_loss/len(loader):.6f}')

tcn_model.eval()
with torch.no_grad():
    tcn_preds = tcn_model(X_test).numpy() * std + mean
    tcn_mae = np.abs(tcn_preds - actuals).mean()
    print(f'\nTCN Test MAE: {tcn_mae:.2f}')

---
## 3 · Transformer for Time Series

Transformers use **self-attention** to look at ALL past timesteps simultaneously (not just a sliding window). This captures complex long-range dependencies but requires more data.

| Architecture | Sequential? | Long-Range | Training Speed | Data Needed |
|-------------|:-----------:|:----------:|:--------------:|:-----------:|
| LSTM | Yes (slow) | Moderate | Slow | Medium |
| TCN | No (parallel) | Good (via dilation) | Fast | Medium |
| Transformer | No (parallel) | Excellent (attention) | Medium | Large |

In [ ]:
# Cell 4 — Simple Transformer Forecaster

class TransformerForecaster(nn.Module):
    def __init__(self, input_dim=1, d_model=64, nhead=4, num_layers=2, output_dim=1, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, 500, d_model) * 0.1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, output_dim)
        # Causal mask: prevent attending to future positions
        self.register_buffer('mask', nn.Transformer.generate_square_subsequent_mask(500))
    
    def forward(self, x):
        seq_len = x.size(1)
        x = self.input_proj(x) + self.pos_encoding[:, :seq_len, :]
        x = self.transformer(x, mask=self.mask[:seq_len, :seq_len])
        return self.fc(x[:, -1, :])  # Last position predicts next

# Train
tf_model = TransformerForecaster()
optimizer = torch.optim.Adam(tf_model.parameters(), lr=0.0005)

for epoch in range(30):
    epoch_loss = 0
    for xb, yb in loader:
        pred = tf_model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d} | Loss: {epoch_loss/len(loader):.6f}')

tf_model.eval()
with torch.no_grad():
    tf_preds = tf_model(X_test).numpy() * std + mean
    tf_mae = np.abs(tf_preds - actuals).mean()

print(f'\n=== Model Comparison ===')
print(f'  LSTM:        MAE = {mae:.2f}')
print(f'  TCN:         MAE = {tcn_mae:.2f}')
print(f'  Transformer: MAE = {tf_mae:.2f}')

---
## 4 · Foundation Models: Chronos & TimesFM

**The 2025-2026 paradigm shift:** Pre-trained foundation models that forecast ANY time series with ZERO training on your data.

| Model | Organization | Approach | Zero-Shot? |
|-------|-------------|---------|:----------:|
| **Chronos** | Amazon | Tokenize values, use T5 language model | ✅ |
| **TimesFM** | Google | Patched time series Transformer | ✅ |
| **Lag-Llama** | ServiceNow | Probabilistic, lags as covariates | ✅ |
| **Moirai** | Salesforce | Universal forecasting Transformer | ✅ |

```python
# Chronos: forecast ANY time series in 3 lines
from chronos import ChronosPipeline

pipeline = ChronosPipeline.from_pretrained("amazon/chronos-t5-small")
forecast = pipeline.predict(
    context=torch.tensor(your_time_series),
    prediction_length=30,   # Predict next 30 steps
    num_samples=20,         # 20 probabilistic samples
)
# Returns: (num_samples, prediction_length) — full forecast distribution!
```

**When to use foundation models vs custom training:**

| Use Foundation Model | Train Custom Model |
|---------------------|-------------------|
| Quick prototyping, baselines | You have 10K+ related series |
| Limited training data | Domain-specific patterns (finance) |
| General forecasting tasks | Strict accuracy requirements |

---
## Knowledge Check

### Q1: LSTM vs TCN
<details><summary>Answer</summary>

LSTMs process sequences step-by-step (sequential, can't parallelize). TCNs use convolutions that process the entire sequence in parallel. TCNs are typically 5-10x faster to train and often match or exceed LSTM accuracy. Use LSTMs only when you need explicit hidden-state dynamics.
</details>

### Q2: Why use a causal mask in the Transformer?
<details><summary>Answer</summary>

Without a causal mask, self-attention at position t can look at positions t+1, t+2, ... (future). This is data leakage. The causal mask sets attention weights to -∞ for future positions, ensuring each position can only attend to past and current positions.
</details>

---
**Next →** `04_production_time_series.ipynb` — Deployment, anomaly detection, and system design